In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
import kagglehub
import os

# ----------------------------------------------------
# STEP 1: Download dataset from Kaggle and set paths
# ----------------------------------------------------

path = kagglehub.dataset_download("armia123/plant-leaf-disease-classification")
print("Path to dataset files:", path)


print("Top-level contents:", os.listdir(path))

TRAIN_DIR = os.path.join(path, "train")   # <-- update if the folder name differs
TEST_DIR = os.path.join(path, "test")     # <-- update if the folder name differs


IMG_SIZE = (128, 128)   # leaf images usually need more detail than 32x32
BATCH_SIZE = 32

# ----------------------------------------------------
# STEP 2: Load Dataset from directories
# ----------------------------------------------------

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    seed=42
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=False
)


class_names = train_ds.class_names
num_classes = len(class_names)
print("Number of classes:", num_classes)
print("Classes:", class_names)

# ----------------------------------------------------
# STEP 3: Visualize Sample Images
# ----------------------------------------------------

plt.figure(figsize=(12, 6))
for images, labels in train_ds.take(1):
    for i in range(min(10, len(images))):
        plt.subplot(2, 5, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[labels[i]], fontsize=8)
        plt.axis('off')
plt.tight_layout()
plt.show()

# ----------------------------------------------------
# STEP 4: Preprocess - Normalize + Augment
# ----------------------------------------------------

normalization_layer = layers.Rescaling(1./255)

data_augmentation = models.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
])

AUTOTUNE = tf.data.AUTOTUNE

def prepare(ds, training=False):
    ds = ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    return ds.cache().prefetch(buffer_size=AUTOTUNE)

train_ds_prepped = prepare(train_ds, training=True)
test_ds_prepped = prepare(test_ds, training=False)

# ----------------------------------------------------
# STEP 5: Build the CNN Model
# ----------------------------------------------------

model = models.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),

    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

model.summary()

# ----------------------------------------------------
# STEP 6: Compile the Model
# ----------------------------------------------------

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ----------------------------------------------------
# STEP 7: Train the Model
# ----------------------------------------------------

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6
)

history = model.fit(
    train_ds_prepped,
    epochs=20,
    steps_per_epoch=50,        # <-- limits each epoch to 250 batches instead of the full dataset
    validation_data=test_ds_prepped,
    validation_steps=20,        # <-- optional: also caps validation batches per epoch
    callbacks=[early_stop, reduce_lr]
)

# ----------------------------------------------------
# STEP 8: Evaluate the Model
# ----------------------------------------------------

test_loss, test_acc = model.evaluate(test_ds_prepped)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")

# ----------------------------------------------------
# STEP 9: Plot Training History
# ----------------------------------------------------

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.legend()

plt.tight_layout()
plt.show()

# ----------------------------------------------------
# STEP 10: Confusion Matrix & Classification Report
# ----------------------------------------------------

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_true = np.concatenate([y.numpy() for x, y in test_ds_prepped], axis=0)
y_pred_probs = model.predict(test_ds_prepped)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(max(8, num_classes * 0.6), max(6, num_classes * 0.5)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names))

# ----------------------------------------------------
# STEP 11: Predict on a Single New Leaf Image
# ----------------------------------------------------

def predict_leaf(img_path):
    img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0) / 255.0
    prediction = model.predict(img_array)
    predicted_class = class_names[np.argmax(prediction)]
    confidence = np.max(prediction) * 100
    print(f"Predicted: {predicted_class} ({confidence:.2f}% confidence)")
    return predicted_class, confidence

# Example usage:
# predict_leaf("dataset/test/Tomato___Early_blight/sample.jpg")

# ----------------------------------------------------
# STEP 12: Save the Model
# ----------------------------------------------------

model.save('plant_disease_cnn_model.keras')
print("Model saved successfully!")

Output hidden; open in https://colab.research.google.com to view.